In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("../data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [4]:
pd.DataFrame(df_ground_truth)

,question,document
0,Is it possible to register since I'm new?,74eb249bbf
1,Can I catch up if I missed the start?,74eb249bbf
2,Will I get disqualified for the diploma if I'm...,74eb249bbf
3,Do I have to turn it in before the window closes?,74eb249bbf
4,Does entering late limit final verification?,74eb249bbf
...,...,...
550,"My environment prevents upgrading components, ...",4b30b918bc
551,Are there external URLs to fetch the build fil...,4b30b918bc
552,Does a connection refusal imply that my identi...,4b30b918bc
553,Should I re-check my access details if the ser...,4b30b918bc


In [5]:
from ingest import load_faq_data, build_index

documents = load_faq_data(file_path="../documents/all_documents.json")

documents_llm = []

for doc in documents:
    if doc["course"] == "llm":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [9]:
documents

[{'course': 'llm',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM . When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.",
  'doc_id': '977bf7786c'},
 {'course': 'llm',
  'section': 'General Course-Related Questions',
  'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
  'answer': 'The zoom link is only published to instructors/

In [7]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["doc_id"]] = doc

In [33]:
doc_idx

{'74eb249bbf': {'course': 'llm',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 '977bf7786c': {'course': 'llm',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM . When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.",
  'doc_id': '977bf7786c'},
 '489dd1c9d9': {'course': 'llm',
  'section': 'General Course-Related Questions',
  'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
  'answer': 'The z

In [56]:
q = ground_truth[25]
q

{'question': 'Is there a way to complete the modules at my own speed and still get the cert?',
 'document': '69d122f12e'}

In [57]:
doc_idx[q['document']]

{'course': 'llm',
 'section': 'General Course-Related Questions',
 'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
 'doc_id': '69d122f12e'}

In [ ]:
from openai import OpenAI

openai_client = OpenAI(
    base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1",
    api_key="your-key"
)

In [87]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
    course='llm',
    model="glm-5.2"
)

In [59]:
q['question']

'Is there a way to complete the modules at my own speed and still get the cert?'

In [60]:
answer = assistant.rag(q['question'])

In [61]:
answer

'No. According to the course info, you can only get a certificate if you finish the course with a “live” cohort. Certificates are **not** awarded for the self-paced mode, because you need to peer-review capstone projects while the course is running.'

In [62]:
assistant.total_cost()

0.0001518

In [68]:
print(answer)

No. According to the course info, you can only get a certificate if you finish the course with a “live” cohort. Certificates are **not** awarded for the self-paced mode, because you need to peer-review capstone projects while the course is running.


In [70]:
doc_id = q["document"]
doc_id

'69d122f12e'

In [71]:
doc_idx[doc_id]

{'course': 'llm',
 'section': 'General Course-Related Questions',
 'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
 'doc_id': '69d122f12e'}

In [72]:
doc_id = q["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'

In [66]:
original_doc

{'course': 'llm',
 'section': 'General Course-Related Questions',
 'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
 'doc_id': '69d122f12e'}

In [67]:
answer_orig

'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'

In [73]:
rag_result = {
    "question": q['question'],
    "answer_llm": answer,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'Is there a way to complete the modules at my own speed and still get the cert?',
 'answer_llm': 'No. According to the course info, you can only get a certificate if you finish the course with a “live” cohort. Certificates are **not** awarded for the self-paced mode, because you need to peer-review capstone projects while the course is running.',
 'answer_orig': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
 'document': '69d122f12e'}

In [75]:
doc_idx

{'74eb249bbf': {'course': 'llm',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 '977bf7786c': {'course': 'llm',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM . When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.",
  'doc_id': '977bf7786c'},
 '489dd1c9d9': {'course': 'llm',
  'section': 'General Course-Related Questions',
  'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
  'answer': 'The z

In [88]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [76]:
q

{'question': 'Is there a way to complete the modules at my own speed and still get the cert?',
 'document': '69d122f12e'}

In [77]:
record = generate_rag_answer(q)
record

{'question': 'Is there a way to complete the modules at my own speed and still get the cert?',
 'answer_llm': 'No, you cannot complete the modules fully at your own speed and still get the certificate. Certificates are only awarded if you finish the course with a “live” cohort. Self-paced mode does not award certificates, partly because you need to peer-review 3 capstone projects, which is only possible while the course is running.',
 'answer_orig': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
 'document': '69d122f12e'}

In [78]:
assistant.total_cost()

0.0003432

In [79]:
assistant.reset_usage()

In [80]:
assistant.total_cost()

0.0

In [86]:
len(ground_truth)

555

In [89]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/555 [00:00<?, ?it/s]

In [84]:
results[:10]

NameError: name 'results' is not defined

In [ ]:
df_results = pd.DataFrame(results)

In [ ]:
df_results.head()

In [ ]:
assistant.total_cost()

In [ ]:
df_results.to_csv("../data/rag-answers-new.csv", index=False)